# 1 · An economy in a notebook: the RBC model

**The story.** Imagine a small island economy. People work, spend part of their income on goods (`c`), and put the rest into machines (`k`) that firms rent to produce everything the island makes (`y`). One day someone invents a better way of organizing production, so technology (`a`) becomes 1% more efficient. That is a *shock* — a surprise. The gain does not vanish immediately: if 1% remains today, 0.9% of it is still there next year, 0.81% the year after, and so on. The boost *persists*, but fades.

**What this notebook does.** Load the model written in [`models/rbc.dyno`](models/rbc.dyno), verify its resting position (steady state), solve an approximate version of it, and trace how output, investment and consumption move after the discovery. Along the way: a five-line inline model to learn the syntax, one possible random future, and a comparison of three islands that differ only in how long a boost lasts.

New to economic models? Run the cells in order. Terms are defined where they first appear and collected in the [glossary](#glossary) at the end.

### The model's cast and its everyday meaning

| Symbol | Everyday meaning | | Symbol | Everyday meaning |
| --- | --- | --- | --- | --- |
| `a` | How efficient technology is | | `w` | Wages paid to workers |
| `y` | Total goods produced (output) | | `r` | Rental fee for machines |
| `c` | Goods people consume | | `epsilon` | The surprise in technology |
| `i` | Machines added this period (investment) | | `beta` | Patience: value of tomorrow vs today |
| `k` | Machines available (capital) | | `delta` | Wear and tear on machines |
| `n` | Fraction of time people work | | `rho` | How long a technology gain lasts |

Open [models/rbc.dyno](models/rbc.dyno) side-by-side while reading: its comments explain every equation in plain English.

In [1]:
import numpy as np
import pandas as pd
from dyno import DynoModel

pd.set_option("display.precision", 4)

## 1. Load the model

`DynoModel` reads the text file, understands the equations, and summarizes what it found. The rendering below — calibration status, variables, and the equations in readable notation — is Dyno's own view of the file. We will only trust the numbers after the checks in the next section.

In [2]:
model = DynoModel("models/rbc.dyno")
model

Component,Count,Symbols
Equations,8,
Endogenous,8,"r, n, k, y, i, c, w, a"
Exogenous,1,epsilon
Parameters,7,"beta, delta, nss, eta, alpha, rho, chi"


### A five-line model first: the minimal recipe

The same language fits in a Python string. This tiny model says: a variable `x` is pulled back toward 0 at rate `rho`, while a random shock `e` of typical size `sigma` pushes it around. The syntax to notice is the complete recipe:

- `[t]` now, `[t-1]` one period ago (`rho * x[t-1]` is the pull toward 0),
- `[~]` the steady state (the value every variable would keep if nothing ever happened),
- `<-` gives a steady state or declares a random process; `=` writes an equation,
- `N(mean, standard deviation)` declares a shock called `e`.

Every later notebook uses exactly these five ingredients.

In [3]:
toy_txt = """
rho   <- 0.85    # speed at which x returns to 0
sigma <- 0.02    # typical size of the surprise e
x[~]  <- 0.0     # steady state: without shocks, x rests at 0
e[t]  <- N(0, sigma)
x[t]  = rho * x[t-1] + e[t]
"""
toy_model = DynoModel(txt=toy_txt)
toy_solution = toy_model.solve()

_, toy_rule = toy_solution.decision_rule.coefficients_as_df()
toy_rule

,x[t-1],e[t]
x[t],0.85,1.0


Each row of the rule table answers: *how does a variable today depend on where the economy was yesterday and on today's surprise?* The single row here just restates the model: `x[t] = 0.85·x[t-1] + 1·e[t]`.

Plotting works on this five-line model exactly as it works on files — every chart in these notebooks is Dyno's built-in plot (hover and zoom are available):

In [4]:
toy_solution.plot(type="percent", variables=["x"], T=20)

alt.Chart(...)

## 2. Verify the resting position: the steady state

Two things protect us from nonsense results: the *steady state* and the *residuals*.

The **steady state** is the economy's resting position: the value of every variable when no shock ever arrives. The model file states one (`a` rests at 1, people work about a third of their time…). A **residual** is what remains of an equation when almost-equal sides differ; if the stated resting position truly satisfies every equilibrium equation, all residuals are essentially zero.

In [5]:
steady_state = pd.DataFrame.from_dict(model.steady_state, orient="index", columns=["Steady state"])
steady_state

,Steady state
a,1.0000
r,0.0402
n,0.3300
k,6.9655
y,0.8757
w,1.8044
i,0.1741
c,0.7015
epsilon,0.0000


In [6]:
residuals = model.residuals
print(f"Maximum absolute residual: {np.max(np.abs(residuals)):.2e}")
assert np.allclose(residuals, 0, atol=1e-6), "Residuals must be close to 0"
print("Steady state passes the numerical check")

Maximum absolute residual: 8.88e-16
Steady state passes the numerical check


## 3. Does the economy settle back down? The uniqueness test

A usable model must settle back to its resting position after any shock, and do so in exactly **one** way — otherwise, "what happens after the discovery" has either no answer or too many. `model.check()` verifies this: it counts the directions in which the linearized economy can move, fast or slow, inward or outward, and verifies there is a single path back to rest. (Economists call this the **Blanchard–Kahn conditions**. The count is what matters; the algebra is Dyno's job.)

In [7]:
model.check(); # doesn't output anything unless the stable solution doesn't exist / isn't unique

## 4. Read the decision rule

`model.solve()` turns the equations into *decision rules*: how far each variable departs from steady state, written as a weighted sum of yesterday's departures and today's surprise. The table below shows how technology (`a`), output (`y`), consumption (`c`) and investment (`i`) respond today:

- the `a[t-1]` column weights yesterday's technology,
- the `epsilon[t]` column weights today's discovery.

Read row `y[t]`: output moves up with both. Comparing rows tells you *who reacts and by how much* — the heart of the model's economics.

In [8]:
solution = model.solve()
_, rule = solution.decision_rule.coefficients_as_df()
rule.loc[["a[t]", "y[t]", "c[t]", "i[t]"], ["a[t-1]", "epsilon[t]"]]

,a[t-1],epsilon[t]
a[t],0.9000,1.0000
y[t],1.2188,1.3542
c[t],0.2191,0.2434
i[t],0.9997,1.1107


## 5. One discovery: the impulse response

An **impulse response** plays out a single surprise in period 1 with *no further luck ever after*, and records the path of every variable. It answers: "the discovery happens now — then what?"

Here, `"percent"` means `100 × (value − steady state) / steady state`: each line is the percentage gap from the resting position. The island's shock has standard deviation 0.009, so technology starts about **0.9%** above rest and fades afterward.

In [9]:
irf = solution.irfs(type="percent", T=40)["epsilon"]
irf[["a", "y", "c", "i", "n"]].head(10)

,a,y,c,i,n
0,0.0000,0.0000,0.0000,0.0000,0.0000
1,0.9000,1.3918,0.3123,5.7407,0.7233
2,0.8100,1.2734,0.3571,4.9649,0.6139
3,0.7290,1.1656,0.3928,4.2787,0.5178
4,0.6561,1.0673,0.4207,3.6726,0.4333
5,0.5905,0.9778,0.4417,3.1378,0.3592
6,0.5314,0.8963,0.4568,2.6665,0.2944
7,0.4783,0.8218,0.4669,2.2519,0.2378
8,0.4305,0.7540,0.4726,1.8876,0.1885
9,0.3874,0.6920,0.4745,1.5681,0.1457


The same numbers, drawn: technology (`a`) jumps and decays. **Investment (`i`) springs some eighteen times farther than consumption on impact (5.7% vs 0.3%)** — people smooth their spending and instead build machines while times are good. The extra machines stay productive for many periods, which is why output (`y`) and consumption stay above rest long after the discovery itself has faded.

In [10]:
solution.plot(type="percent", variables=["a", "y", "c", "i"], T=40)

alt.Chart(...)

## 6. One possible future: a simulation

The impulse response holds future luck at zero; the **simulation** instead *rolls the dice* every period. Each run is one possible future — a different throw of the dice changes it. The `rng=` seed fixes the dice, so this exact future is reproducible.

In [11]:
simulation = solution.simulate(T=120, mode="random", units="level", rng=7)
simulation.to_df()[["a", "y", "c", "i", "n"]].head(10)

,a,y,c,i,n
0,1.0000,0.8757,0.7015,0.1741,0.3300
1,1.0000,0.8757,0.7015,0.1741,0.3300
2,1.0027,0.8793,0.7022,0.1771,0.3307
3,1.0000,0.8757,0.7017,0.1740,0.3300
4,0.9920,0.8648,0.6997,0.1651,0.3278
5,0.9887,0.8602,0.6984,0.1618,0.3271
6,0.9809,0.8494,0.6959,0.1535,0.3252
7,0.9833,0.8524,0.6954,0.1570,0.3261
8,0.9971,0.8707,0.6979,0.1728,0.3299
9,0.9929,0.8651,0.6969,0.1682,0.3288


One drawn history (in percent from steady state). These wobbles are the business cycle *this* model generates: persistent, uneven, and driven entirely by the dice — there is no other source of disturbance by construction.

In [12]:
simulation.plot(units="percent", variables=["y", "c", "i"], T=120)

alt.Chart(...)

### The model's promises versus one drawn history

The solution's **moments** describe what the model promises on average over all possible dice throws — here, the standard deviation of each variable. A single 120-period draw will differ by luck; longer draws get closer to the promised values.

In [13]:
cov_uncond, _ = solution.decision_rule.moments()
variable_names = model.symbols["endogenous"]
model_std = pd.Series(np.sqrt(np.diag(cov_uncond)), index=variable_names, name="Model promise")
sample_std = simulation.to_df()[variable_names].std().rename("120-period draw")
pd.concat([model_std, sample_std], axis=1).loc[["y", "c", "i", "n"]]

,Model promise,120-period draw
y,0.0122,0.0284
c,0.0022,0.0140
i,0.0100,0.0190
n,0.0024,0.0045


## 7. Three islands, the same discovery

One parameter — `rho` — controls how long a technology gain lasts (`0.98`: an echo that lasts decades; `0.70`: mostly a memory within a couple of years). `model.variants(...)` rebuilds the file's model with each value, without editing the source text, and one chained pipeline takes all variants from calibration to plot.

In [14]:
variant_irfs = (
    model.variants(rho=[0.70, 0.90, 0.98])
    .steady()
    .check()
    .solve()
    .simulate(mode="irf", T=40, units="percent")
)
variant_irfs.plot(variables=["y"], T=40, units="percent")

alt.Chart(...)

<a id="glossary"></a>
## Glossary

<details><summary>Everyday-language definitions of the terms used above</summary>

- **Model** — a set of equations describing how an economy evolves; the `.dyno` file is one.
- **Endogenous** — decided *inside* the model (output, consumption, …). The model's equations pin them down.
- **Exogenous / shock** — given from *outside* and not explained by the model; the discovery `epsilon` is one.
- **Innovation** — another word for the shock: the genuinely new part of a surprise.
- **Steady state** — the resting position every variable takes and keeps when no shock ever arrives.
- **Residual** — what remains when the two sides of an equation don't quite match; near-zero residuals mean consistency.
- **Persistence** — the fraction of a shock that survives into the next period; `rho` above.
- **Impulse response** — the variable paths after one shock in period 1 with luck at zero afterwards.
- **Decision rule** — the solved model, rewritten as "today = weights × yesterday + weights × today's surprise".
- **First-order approximation** — a linear simplified copy of the model, accurate for small deviations from rest. (Dyno also solves exact nonlinear paths; see notebook 2.)
- **Blanchard–Kahn conditions** — the uniqueness test of the linearized system (section 3).
- **Simulation** — one dice-drawn future obtained by giving the shocks random values every period.
- **Moments** — averages over all possible dice throws (variances, correlations), *not* over history.
- **Deterministic vs stochastic** — a model is *stochastic* when shocks are random (this notebook); *deterministic* when everything, including the news, is announced in advance (notebook 2).
- **Chart units** — `level` shows the plain value; `deviation` shows the difference from rest; `percent` (used here) shows `100 × deviation / steady state`, comparable across variables of different sizes.
- **DSGE / RBC** — family names for models like this one (Dynamic Stochastic General Equilibrium / Real Business Cycle). "Dynamic": over time. "Stochastic": has shocks. "General equilibrium": every market clears at once. "Real": quantities, not money.
</details>

**Takeaway.** The workflow is always the same: model text → verify resting position and uniqueness → solve → impulse responses, simulated futures, and comparisons across calibrations. A linear solution is most informative near the resting position; notebook 2 solves an exact nonlinear path when the moves are large, and notebook 3 double-checks the same island economy written in Dynare's file format.